# 第 21 课｜规模变大以后，瓶颈会跑到哪里？

一个在 tiny graph 上看起来很均衡的设计，规模变大后可能被 memory、queue，或者少数高流量 neuron 限制。

今天只问一个问题：

> **在某个具体 workload scale 下，怎样判断哪一个 stage 正在限制系统？**

本课主要新概念：**bottleneck 是“相对于 capacity，utilization 最高的 stage”。**

## 1. 概念账本

**已经知道：** latency、throughput、bandwidth、FIFO backpressure、DDR、sparse fan-out。

**今天学习：** **utilization（利用率）**与随 scale 改变的 **bottleneck（瓶颈）**。

**只预告：** 实测 10K/50K MaleCNS、bank conflict、telemetry，以及 cache / multiple synapse engines 等优化。

## 2. bottleneck 是关系，不是永久标签

对一个 stage：

[
utilization = rac{demand}{capacity}
]

utilization 接近或超过 1.0，说明几乎没有 headroom，甚至已经超出能力。

当 graph、event rate、memory pattern 或 architecture 改变时，bottleneck 可以移动。“DDR 永远是瓶颈”不是 specification。

## 3. pipeline 图

```mermaid
flowchart LR
  A["spike source"] --> B["FIFO"]
  B --> C["synapse lookup"]
  C --> D["DDR / storage"]
  D --> E["target update"]
```

每个 stage 都可能有不同 capacity 与 demand。

## 4. Run：一个 synthetic scale study

下面都是**教学数字**，不是 FPGA 实测结果。目的只是练习分析方法。

In [ ]:
capacities = {
    "fifo": 10.0,
    "lookup": 8.0,
    "memory": 6.0,
    "update": 9.0,
}

cases = {
    "small": {"fifo": 2.0, "lookup": 2.5, "memory": 2.0, "update": 2.2},
    "medium": {"fifo": 5.0, "lookup": 5.5, "memory": 5.7, "update": 4.8},
    "large": {"fifo": 7.0, "lookup": 6.5, "memory": 7.2, "update": 6.8},
}

for name, demand in cases.items():
    utilization = {
        stage: demand[stage] / capacities[stage]
        for stage in capacities
    }
    bottleneck = max(utilization, key=utilization.get)
    print(name, "bottleneck:", bottleneck,
          "utilization:", round(utilization[bottleneck], 2))

## 5. Observe

真正比较的是**demand 相对于 capacity**，而不是 raw demand 谁最大。

在 large teaching case 中，memory demand 超过 toy memory capacity。这告诉我们该从哪里调查，但还不能直接证明原因一定是 DDR bandwidth，也可能涉及 access pattern、bank conflict 或 traffic hotspot。

## 6. hotspot 是“不均匀的工作”

两个 network 可以拥有相同 edge 总数，却有非常不同的 traffic distribution。少数 high-fanout 或 high-rate source 可能制造 queue pressure 或 bank conflict。

所以 “network size” 不是完整的 performance 描述，distribution 也重要。

## 7. 先测量，再优化

RMD-022 故意排在 correctness baseline 之后。

cache、banking、lazy update 或更多 engine 都不自动等于更好。应该先建立：

1. correct baseline；
2. telemetry 与明确 workload；
3. 实际观察到的 limiting resource；
4. 同 workload 的 before/after benchmark。

## 8. Try It

只提高 large case 的 `lookup` capacity。先预测 bottleneck 是否改变。

再只提高 memory capacity。哪一个修改会改变诊断？

## 9. 作业

[第 21 课作业：找出 utilization 最高的 stage](../../exercises/zh/21_scaling_bottlenecks.ipynb)

## 10. AI Task

给 AI 一张 utilization table，让它提出 hottest stage 的三个可能原因。要求它把这些写成 hypothesis，而不是伪装成 measurement。

## 11. Human Check

解释为什么 bottleneck 会随 scale 改变。为什么 “raw workload 最大” 不一定等于 “utilization 最高”？选择 optimization 前需要什么证据？

## 12. Engineering Handoff

对应 `RMD-019~022`、`MOD-014 telemetry` 与 `P-001~P-008`。正式性能结论必须来自定义清楚的 workload 与实测 report；本课数字只属于 teaching fixture。

## 13. Project Trace

- Lesson：`LSN-021`
- 映射：`RMD-019 / RMD-020 / RMD-021 / RMD-022`
- 需求路径：`TRACE-P-001`、`TRACE-F-001`
- optimization 前 correctness oracle：`T-016`
- performance metrics：`P-001~P-008`

## 14. Exit Ticket

面对多个 stage 的 demand 与 capacity，你能够计算 utilization、找出当前 bottleneck candidate，并解释为什么换一个 scale 后诊断可能变化。